In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("fraud-streaming") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.1.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark session ready")

✅ Spark session ready


In [2]:
from pyspark.sql.functions import from_json, col, window, sum as _sum, count, avg
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("type", StringType()),
    StructField("amount", DoubleType()),
    StructField("nameOrig", StringType()),
    StructField("nameDest", StringType()),
    StructField("oldbalanceOrg", DoubleType()),
    StructField("newbalanceOrig", DoubleType()),
    StructField("isFraud", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("source", StringType()),
])

raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

print("✅ Streaming reader defined")

✅ Streaming reader defined


In [3]:
checkpoint_path = "s3a://silver/checkpoints/streaming/"
output_path = "s3a://silver/streaming/transactions/"

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("path", output_path) \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Streaming query started — writing to silver Delta table")
print(f"Status: {query.status}")

✅ Streaming query started — writing to silver Delta table
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [6]:
print(query.status)
print(query.lastProgress)

{'message': 'Getting offsets from KafkaV2[Subscribe[transactions]]', 'isDataAvailable': False, 'isTriggerActive': True}
None


In [8]:
print(query.status)
print(query.lastProgress)

{'message': 'Terminated with exception: org.apache.kafka.common.errors.TimeoutException: Timed out waiting for a node assignment. Call: describeTopics', 'isDataAvailable': False, 'isTriggerActive': False}
None


In [9]:
query.stop()

In [10]:
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/streaming/") \
    .option("path", "s3a://silver/streaming/transactions/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Stream started — status: {query.status}")

✅ Stream started — status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [11]:
print(query.status)
print(query.lastProgress)

{'message': 'Getting offsets from KafkaV2[Subscribe[transactions]]', 'isDataAvailable': False, 'isTriggerActive': True}
None


In [12]:
print(query.status)
print(query.lastProgress)

{'message': 'Getting offsets from KafkaV2[Subscribe[transactions]]', 'isDataAvailable': False, 'isTriggerActive': True}
None


In [13]:
query.stop()

In [14]:
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/streaming/") \
    .option("path", "s3a://silver/streaming/transactions/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Stream started — status: {query.status}")

✅ Stream started — status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [15]:
print(query.status)
print(query.lastProgress)

{'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}
{'id': '4d48a204-fcaf-4e9f-8ac4-ba265b7db7c3', 'runId': '0e043edf-88b8-459b-832e-8327cdb3d220', 'name': None, 'timestamp': '2026-05-26T17:58:10.000Z', 'batchId': 8, 'numInputRows': 99, 'inputRowsPerSecond': 9.9, 'processedRowsPerSecond': 38.976377952755904, 'durationMs': {'addBatch': 2377, 'commitOffsets': 87, 'getBatch': 0, 'latestOffset': 8, 'queryPlanning': 9, 'triggerExecution': 2540, 'walCommit': 58}, 'stateOperators': [], 'sources': [{'description': 'KafkaV2[Subscribe[transactions]]', 'startOffset': {'transactions': {'2': 1593, '5': 1619, '4': 1617, '7': 1567, '1': 1577, '3': 1501, '6': 1581, '0': 1553}}, 'endOffset': {'transactions': {'2': 1605, '5': 1630, '4': 1629, '7': 1578, '1': 1588, '3': 1517, '6': 1594, '0': 1566}}, 'latestOffset': {'transactions': {'2': 1605, '5': 1630, '4': 1629, '7': 1578, '1': 1588, '3': 1517, '6': 1594, '0': 1566}}, 'numInputRows': 99, 'inputRowsPerSecond': 9.9

In [16]:
df = spark.read.format("delta").load("s3a://silver/streaming/transactions/")
print(f"✅ Rows in silver streaming table: {df.count()}")
df.show(5)

✅ Rows in silver streaming table: 1305
+--------------+--------+-------+--------+--------+-------------+--------------+-------+--------------------+---------+--------------------+
|transaction_id|    type| amount|nameOrig|nameDest|oldbalanceOrg|newbalanceOrig|isFraud|           timestamp|   source|     kafka_timestamp|
+--------------+--------+-------+--------+--------+-------------+--------------+-------+--------------------+---------+--------------------+
|       T344884|CASH_OUT|2738.56|C2813106|C4549325|    399908.32|      511044.4|      0|2026-05-26T17:57:...|synthetic|2026-05-26 17:57:...|
|       T171697|TRANSFER|4541.53|C1345411|C2200294|    719167.12|     975987.81|      0|2026-05-26T17:57:...|synthetic|2026-05-26 17:57:...|
|       T366097| CASH_IN|1042.88|C4505947|C4116070|    204117.43|     853590.05|      0|2026-05-26T17:57:...|synthetic|2026-05-26 17:57:...|
|       T495086|TRANSFER|3800.12|C4529508|C4414494|    845273.45|      270746.4|      0|2026-05-26T17:57:...|synthe

In [17]:
from pyspark.sql.functions import to_timestamp, window, col, sum as _sum, count, avg

# Stop current query first
query.stop()

# Re-read stream with watermark
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

# Add proper timestamp and watermark for late data (10 minute tolerance)
with_watermark = parsed \
    .withColumn("event_time", to_timestamp(col("timestamp"))) \
    .withWatermark("event_time", "10 minutes")

# Sliding window aggregations per card per 1h window
windowed = with_watermark.groupBy(
    window(col("event_time"), "1 hour", "10 minutes"),
    col("nameOrig")
).agg(
    count("transaction_id").alias("tx_count_1h"),
    _sum("amount").alias("total_amount_1h"),
    avg("amount").alias("avg_amount_1h"),
    _sum("isFraud").alias("fraud_count_1h")
)

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Windowed stream started: {query_windowed.status}")

✅ Windowed stream started: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [18]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Windowed features rows: {df_w.count()}")
df_w.show(5, truncate=False)

Windowed features rows: 0
+------+--------+-----------+---------------+-------------+--------------+
|window|nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------+--------+-----------+---------------+-------------+--------------+
+------+--------+-----------+---------------+-------------+--------------+



In [19]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Windowed features rows: {df_w.count()}")
df_w.show(5, truncate=False)

Windowed features rows: 0
+------+--------+-----------+---------------+-------------+--------------+
|window|nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------+--------+-----------+---------------+-------------+--------------+
+------+--------+-----------+---------------+-------------+--------------+



In [20]:
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

✅ Checkpoint cleared


In [21]:
query_windowed.stop()

In [24]:
query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

In [25]:
df_check = spark.read.format("delta") \
    .load("s3a://silver/streaming/windowed_features/")

df_check.show(truncate=False)

+------+--------+-----------+---------------+-------------+--------------+
|window|nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------+--------+-----------+---------------+-------------+--------------+
+------+--------+-----------+---------------+-------------+--------------+

